#Lab 11: Simulated Annealing and Constraint Satisfaction Problems

##Overview

In this lab, you will implement:

- Simulated Annealing for optimization
- Node Consistency for unary constraints
- Arc Consistency for binary constraints
- AC-3 algorithm for full constraint propagation

All algorithms must read from input files and produce outputs using a fixed OutputWriter.

##Input File Format

You will use two input files.

###Simulated Annealing Input File

Each line contains a parameter in key-value format:

key = value

Required parameters:
T (initial temperature)
cooling_rate
max_iterations
initial_state

###CSP Input File

This file contains two sections.

First section: variable domains

Format:
Variable: value1,value2,value3

Second section starts with:

constraints:

Each line after this defines a binary constraint:

X != Y

##Step 01: Make Imports

In [25]:
import random
import math
from collections import deque

##Step 02: Input Handler (Do NOT Modify)

In [26]:
class InputHandler:
    def __init__(self, path):
        self.path = path

    def load_sa(self):
        config = {}
        with open(self.path, "r") as f:
            for line in f:
                if "=" in line:
                    k, v = line.strip().split("=")
                    config[k.strip()] = float(v.strip())

        config["initial_state"] = int(config["initial_state"])
        config["max_iterations"] = int(config["max_iterations"])
        return config

    def load_csp(self):
        domains = {}
        constraints = []
        reading_constraints = False

        with open(self.path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                if line.lower().startswith("constraints"):
                    reading_constraints = True
                    continue

                if not reading_constraints:
                    var, dom = line.split(":")
                    domains[var.strip()] = list(map(int, dom.split(",")))
                else:
                    constraints.append(tuple(line.split("!=")))

        return domains, constraints

##Step 03: Simulated Annealing

In [27]:
class SimulatedAnnealing:
    def __init__(self, config):
        self.state = config["initial_state"]
        self.T = config["T"]
        self.cooling_rate = config["cooling_rate"]
        self.max_iterations = config["max_iterations"]

    def cost(self, x):
        return -x  

    def neighbor(self, x):
        delta = random.randint(-5, 5)
        return max(1, x + delta)

    def run(self):
        log = []
        current_state = self.state
        current_cost = self.cost(current_state)
        best_state = current_state
        best_cost = current_cost

        for i in range(self.max_iterations):
            next_state = self.neighbor(current_state)
            next_cost = self.cost(next_state)
            
            delta_cost = next_cost - current_cost
            
            if delta_cost < 0 or random.random() < math.exp(-delta_cost / self.T):
                current_state = next_state
                current_cost = next_cost
            
            if current_cost < best_cost:
                best_state = current_state
                best_cost = current_cost
            
            log.append({
                "iteration": i,
                "temperature": self.T,
                "current_state": current_state,
                "best_state": best_state
            })
            
            self.T *= self.cooling_rate

        return log

##Step 04: Node Consistency

In [28]:
class NodeConsistency:
    def __init__(self, domains):
        self.domains = domains

    def enforce(self):
        return self.domains

##Step 05: Arc Consistency

In [29]:
class ArcConsistency:
    def __init__(self, domains, constraints):
        self.domains = domains
        self.constraints = constraints

    def revise(self, Xi, Xj):
        revised = False
        constraint_type = "!="
        
        Xi_domain = self.domains[Xi]
        Xj_domain = self.domains[Xj]
        
        to_remove = []
        for x in Xi_domain:
            has_consistent = False
            for y in Xj_domain:
                if constraint_type == "!=" and x != y:
                    has_consistent = True
                    break
            
            if not has_consistent:
                to_remove.append(x)
                revised = True
        
        for x in to_remove:
            Xi_domain.remove(x)
        
        return revised

    def enforce(self):
        for Xi, Xj in self.constraints:
            self.revise(Xi.strip(), Xj.strip())
            self.revise(Xj.strip(), Xi.strip())
        
        return self.domains

##Step 06: AC-3

In [ ]:
class AC3:
    def __init__(self, domains, constraints):
        self.domains = domains
        self.constraints = constraints

    def revise(self, Xi, Xj):
        revised = False
        
        Xi_domain = self.domains[Xi]
        Xj_domain = self.domains[Xj]
        
        to_remove = []
        for x in Xi_domain:
            has_consistent = False
            for y in Xj_domain:
                if x != y:
                    has_consistent = True
                    break
            
            if not has_consistent:
                to_remove.append(x)
                revised = True
        
        for x in to_remove:
            Xi_domain.remove(x)
        
        return revised

    def ac3(self):
        queue = deque()
        
        for (Xi, Xj) in self.constraints:
            Xi = Xi.strip()
            Xj = Xj.strip()
            queue.append((Xi, Xj))
            queue.append((Xj, Xi))  # both directions needed for symmetric != constraint
        
        while queue:
            Xi, Xj = queue.popleft()
            
            if self.revise(Xi, Xj):
                for (Xk, Xl) in self.constraints:
                    Xk = Xk.strip()
                    Xl = Xl.strip()
                    
                    if Xl == Xi and Xk != Xj:
                        queue.append((Xk, Xi))
                    elif Xk == Xi and Xl != Xj:
                        queue.append((Xl, Xi))

        return self.domains

##Step 07: Output Writer (Do NOT modify)

In [31]:
class OutputWriter:
    def __init__(self, path):
        self.path = path

    def write(self, title, data):
        with open(self.path, "a") as f:
            f.write("\n=== " + title + " ===\n")
            f.write(str(data) + "\n")

##Step 08: Main Method (Do NOT Modify)

In [32]:
def main():
    sa_config = InputHandler("sa_input.txt").load_sa()
    domains, constraints = InputHandler("csp_input.txt").load_csp()

    sa = SimulatedAnnealing(sa_config)
    sa_result = sa.run()

    nc = NodeConsistency(domains)
    nc_result = nc.enforce()

    ac = ArcConsistency(nc_result, constraints)
    ac_result = ac.enforce()

    ac3 = AC3(domains, constraints)
    ac3_result = ac3.ac3()

    writer = OutputWriter("output.txt")

    print(sa_result)
    print(nc_result)
    print(ac_result)
    print(ac3_result)

    writer.write("SA", sa_result)
    writer.write("NODE", nc_result)
    writer.write("ARC", ac_result)
    writer.write("AC3", ac3_result)


if __name__ == "__main__":
    main()

[{'iteration': 0, 'temperature': 250.0, 'current_state': 34, 'best_state': 37}, {'iteration': 1, 'temperature': 232.5, 'current_state': 38, 'best_state': 38}, {'iteration': 2, 'temperature': 216.22500000000002, 'current_state': 35, 'best_state': 38}, {'iteration': 3, 'temperature': 201.08925000000002, 'current_state': 33, 'best_state': 38}, {'iteration': 4, 'temperature': 187.01300250000003, 'current_state': 35, 'best_state': 38}, {'iteration': 5, 'temperature': 173.92209232500002, 'current_state': 36, 'best_state': 38}, {'iteration': 6, 'temperature': 161.74754586225004, 'current_state': 41, 'best_state': 41}, {'iteration': 7, 'temperature': 150.42521765189255, 'current_state': 40, 'best_state': 41}, {'iteration': 8, 'temperature': 139.89545241626007, 'current_state': 42, 'best_state': 42}, {'iteration': 9, 'temperature': 130.10277074712187, 'current_state': 47, 'best_state': 47}, {'iteration': 10, 'temperature': 120.99557679482334, 'current_state': 46, 'best_state': 47}, {'iteration'